# 04 - Avaliação de qualidade com LLM-as-a-judge no MLflow

Caso prático: avaliar o agente de atendimento LangGraph do notebook anterior. Em vez de ler as respostas uma a uma, usamos **juízes**: outros LLMs que recebem a pergunta, a resposta do agente e um critério, e devolvem `pass`/`fail` (ou `yes`/`no`) com uma justificativa.

O notebook faz, nesta ordem:

1. garante os prompts no Prompt Registry (reaproveita a versão 1 se já existir, como no notebook 03);
2. registra o agente `agente-langgraph` com o código de `agente/agente_langgraph.py`, ligado aos prompts;
3. invoca o agente com um conjunto de perguntas e mostra os traces;
4. define o conjunto de avaliação, com a resposta esperada de cada pergunta;
5. roda `mlflow.genai.evaluate` com juízes prontos do MLflow e um juiz próprio, criado com `make_judge`;
6. mostra as métricas e as respostas reprovadas, com a justificativa do juiz.

Os juízes usam o modelo `openai:/gpt-5-mini`, pelo mesmo `OPENAI_API_KEY` do `.env`. Cada juiz faz uma chamada ao LLM por linha do conjunto, então a avaliação gasta tokens.

## 1. Configuração

A avaliação fica num experimento separado, `agente-avaliacao`. O agente e os juízes usam o modelo de `OPENAI_MODEL`, e os juízes são chamados com o formato `openai:/<modelo>` do MLflow.

In [1]:
import os
from pathlib import Path

import mlflow
import mlflow.genai
import pandas as pd
import yaml
from dotenv import find_dotenv, load_dotenv
from IPython.display import Code, display
from langchain_core.messages import HumanMessage
from mlflow.genai import make_judge
from mlflow.genai.scorers import Correctness, Guidelines, RelevanceToQuery, Safety

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENTO = "agente-avaliacao"
NOME_MODELO = "agente-langgraph"
NOME_PROMPT_CLASSIFICADOR = "langgraph-classificador"
NOME_PROMPT_RESPONDER = "langgraph-responder"

MODELO = os.getenv("OPENAI_MODEL", "gpt-5-mini")
JUIZ = f"openai:/{MODELO}"

ARQUIVO_CODIGO = Path("agente/agente_langgraph.py")
PASTA_PROMPTS = Path("prompts")
PASTA_PROMPTS.mkdir(exist_ok=True)

experimento = mlflow.set_experiment(EXPERIMENTO)
print(f"Experimento '{experimento.name}' (id {experimento.experiment_id})")

Experimento 'agente-avaliacao' (id 2)


## 2. Prompts do agente

Os YAMLs são os mesmos do notebook 03. `garantir_prompt` registra a versão 1 só se ela ainda não existir. O código do agente aponta para a versão 1 (`prompts:/<nome>/1`), então ela precisa existir.

In [2]:
YAML_CLASSIFICADOR = """
commit: "Classifica a pergunta em devolucao, entrega ou outro"
system: |
  Você classifica perguntas de clientes de uma loja. Responda apenas com uma palavra: devolucao, entrega ou outro.
  Use devolucao para trocas, devoluções e reembolsos. Use entrega para prazos e rastreio de pedidos.
  Use outro para qualquer outro assunto.
user: |
  {{ pergunta }}
"""

YAML_RESPONDER = """
commit: "Responde à pergunta de acordo com a categoria"
system: |
  Você é o atendente virtual da {{ nome_loja }}. Responda em português, de forma cordial,
  em até 3 frases. Assunto da pergunta: {{ categoria }}.
user: |
  {{ pergunta }}
"""


def garantir_prompt(nome: str, conteudo: str) -> str:
    uri = f"prompts:/{nome}/1"
    if mlflow.genai.load_prompt(uri, allow_missing=True) is not None:
        print(f"Reaproveitando {uri}")
        return uri

    caminho = PASTA_PROMPTS / f"{nome}.yaml"
    caminho.write_text(conteudo, encoding="utf-8")
    dados = yaml.safe_load(conteudo)
    mensagens = [
        {"role": "system", "content": dados["system"]},
        {"role": "user", "content": dados["user"]},
    ]
    versao = mlflow.genai.register_prompt(name=nome, template=mensagens, commit_message=dados["commit"])
    assert versao.version == 1, "O agente aponta para a versão 1 do prompt."
    print(f"Registrado {uri}")
    return uri


URI_CLASSIFICADOR = garantir_prompt(NOME_PROMPT_CLASSIFICADOR, YAML_CLASSIFICADOR)
URI_RESPONDER = garantir_prompt(NOME_PROMPT_RESPONDER, YAML_RESPONDER)

Reaproveitando prompts:/langgraph-classificador/1
Reaproveitando prompts:/langgraph-responder/1


## 3. Registrando o agente

O código do grafo é o mesmo do notebook 03 (`agente/agente_langgraph.py`). O registro grava uma versão de `agente-langgraph`, com os prompts ligados ao modelo. O `info.model_id` identifica esta versão e é usado na avaliação, para que os resultados fiquem associados a ela.

In [3]:
display(Code(filename=str(ARQUIVO_CODIGO), language="python"))

"""Agente LangGraph versionado com MLflow (models from code).

O grafo tem três nós:
- classificar: escolhe a categoria da pergunta (usa o prompt `langgraph-classificador`);
- responder: responde à pergunta de devolução ou entrega (usa o prompt `langgraph-responder`);
- fora_do_escopo: responde com uma mensagem fixa quando a pergunta não é atendida.

As URIs dos prompts ficam fixas aqui, então cada versão do agente usa versões conhecidas.
O `autolog` do LangChain grava um trace a cada invocação.
"""

import os

import mlflow
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from mlflow.models import set_model

PROMPT_CLASSIFICADOR = "prompts:/langgraph-classificador/1"
PROMPT_RESPONDER = "prompts:/langgraph-responder/1"
NOME_LOJA = "Loja Exemplo"
CATEGORIAS_ATENDIDAS = ("devolucao", "entrega")

mlflow.langchain.autolog()

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), api_key=os.getenv("OPENAI_API_KEY"))

prompt_classificador = mlflow.genai.load_prompt(PROMPT_CLASSIFICADOR)
prompt_responder = mlflow.genai.load_prompt(PROMPT_RESPONDER)


class Estado(MessagesState):
    categoria: str


def pergunta_atual(estado: Estado) -> str:
    return estado["messages"][-1].content


def classificar(estado: Estado) -> dict:
    mensagens = prompt_classificador.format(pergunta=pergunta_atual(estado))
    categoria = llm.invoke(mensagens).content.strip().lower()
    if categoria not in CATEGORIAS_ATENDIDAS:
        categoria = "outro"
    return {"categoria": categoria}


def responder(estado: Estado) -> dict:
    mensagens = prompt_responder.format(
        nome_loja=NOME_LOJA,
        categoria=estado["categoria"],
        pergunta=pergunta_atual(estado),
    )
    resposta = llm.invoke(mensagens)
    return {"messages": [AIMessage(content=resposta.content)]}


def fora_do_escopo(estado: Estado) -> dict:
    texto = "Só consigo ajudar com devoluções e entregas. Vou encaminhar sua pergunta a um atendente."
    return {"messages": [AIMessage(content=texto)]}


def rotear(estado: Estado) -> str:
    return "responder" if estado["categoria"] in CATEGORIAS_ATENDIDAS else "fora_do_escopo"


construtor = StateGraph(Estado)
construtor.add_node("classificar", classificar)
construtor.add_node("responder", responder)
construtor.add_node("fora_do_escopo", fora_do_escopo)

construtor.add_edge(START, "classificar")
construtor.add_conditional_edges("classificar", rotear, ["responder", "fora_do_escopo"])
construtor.add_edge("responder", END)
construtor.add_edge("fora_do_escopo", END)

grafo_agente = construtor.compile()

set_model(grafo_agente)

In [4]:
EXEMPLO = {"messages": [{"role": "user", "content": "Posso devolver um produto depois de 10 dias?"}]}

with mlflow.start_run(run_name="agente-langgraph-para-avaliacao"):
    mlflow.log_params({
        "prompt_classificador": URI_CLASSIFICADOR,
        "prompt_responder": URI_RESPONDER,
        "codigo": str(ARQUIVO_CODIGO),
    })
    info = mlflow.langchain.log_model(
        lc_model=str(ARQUIVO_CODIGO.resolve()),
        name="agente",
        registered_model_name=NOME_MODELO,
        prompts=[URI_CLASSIFICADOR, URI_RESPONDER],
        input_example=EXEMPLO,
    )

print(f"Agente registrado: {NOME_MODELO} versão {info.registered_model_version} (model_id {info.model_id})")

modelo = mlflow.pyfunc.load_model(info.model_uri)


def texto_da_resposta(resposta) -> str:
    estado = resposta[0] if isinstance(resposta, list) else resposta
    return estado["messages"][-1]["content"]


def responder(pergunta: str) -> str:
    """Função que a avaliação chama para gerar a resposta do agente."""
    return texto_da_resposta(modelo.predict({"messages": [{"role": "user", "content": pergunta}]}))

2026/11/16 22:50:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/16 22:50:47 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-2bde2db730ca40a28c5be790f17956eb
2026/11/16 22:50:47 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/11/16 22:50:54 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


Agente registrado: agente-langgraph versão 3 (model_id m-2bde2db730ca40a28c5be790f17956eb)


Registered model 'agente-langgraph' already exists. Creating a new version of this model...
Created version '3' of model 'agente-langgraph'.


## 4. Conjunto de avaliação

Cada item tem `inputs`, a pergunta que chega ao agente, e `expectations`, a resposta esperada. A resposta esperada é usada pelo juiz `Correctness`. Ela descreve o comportamento correto, e não um texto exato.

In [5]:
DATASET = [
    {
        "inputs": {"pergunta": "Posso devolver um produto depois de 10 dias?"},
        "expectations": {"expected_response": "Informa que o direito de arrependimento é de 7 dias corridos para compras online e orienta a consultar a política de devoluções ou falar com um atendente."},
    },
    {
        "inputs": {"pergunta": "Quanto tempo leva para chegar uma entrega em Curitiba?"},
        "expectations": {"expected_response": "Informa um prazo em dias úteis para o frete padrão e pede o CEP ou o número do pedido para confirmar."},
    },
    {
        "inputs": {"pergunta": "Quero trocar um produto personalizado com gravação."},
        "expectations": {"expected_response": "Informa que produtos personalizados não podem ser trocados, exceto em caso de defeito."},
    },
    {
        "inputs": {"pergunta": "Qual é a capital da França?"},
        "expectations": {"expected_response": "Informa que só ajuda com devoluções e entregas e encaminha a pergunta a um atendente."},
    },
]

print(f"{len(DATASET)} casos no conjunto de avaliação.")

4 casos no conjunto de avaliação.


## 5. Traces das invocações

Antes de avaliar, invocamos o agente para cada pergunta. Cada chamada gera um trace no experimento, como no notebook 03. A avaliação também gera traces, então esta etapa mostra que a chamada direta já é rastreada.

In [6]:
for caso in DATASET:
    pergunta = caso["inputs"]["pergunta"]
    print(f"{pergunta}\n→ {responder(pergunta)}\n")

mlflow.flush_trace_async_logging()
traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=50)
print(f"Traces registrados no experimento: {len(traces)}")

Posso devolver um produto depois de 10 dias?
→ Depende: para compras feitas a distância (online) há normalmente o direito de arrependimento de 7 dias a partir do recebimento; após esse prazo só aceitamos devolução se houver defeito/vício ou conforme nossa política interna. Se quiser, me informe o número do pedido e o motivo da devolução que verifico o caso pra você.

Quanto tempo leva para chegar uma entrega em Curitiba?
→ Normalmente o prazo para entrega em Curitiba varia conforme o frete e disponibilidade: cerca de 1–2 dias úteis no expresso e 3–7 dias úteis no padrão, além de 1–2 dias úteis para separação/expedição. Pode variar por produto, estoque e endereço — quer que eu verifique pelo CEP ou número do pedido para um prazo exato?

Quero trocar um produto personalizado com gravação.
→ Normalmente produtos personalizados com gravação não são passíveis de troca, exceto em caso de defeito de fabricação ou erro no pedido por parte da loja. Por favor envie o número do pedido e fotos do 

/tmp/ipykernel_171591/1980172580.py:6: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=50)


## 6. Juízes

Usamos cinco juízes:

- `Correctness`: a resposta está de acordo com a resposta esperada? Usa `expected_response`.
- `RelevanceToQuery`: a resposta trata da pergunta?
- `Safety`: a resposta é segura, sem conteúdo ofensivo ou danoso?
- `Guidelines`: a resposta segue uma regra escrita em linguagem natural. Aqui, a regra é ter no máximo 3 frases.
- `tom_atendimento`: um juiz próprio, criado com `make_judge`. Ele usa instruções escritas por nós e devolve `True` ou `False`.

Os juízes do MLflow recebem a pergunta, a resposta e, quando é o caso, a resposta esperada, e devolvem uma justificativa junto com o veredito.

In [7]:
tom_atendimento = make_judge(
    name="tom_atendimento",
    instructions=(
        "Avalie se a resposta {{ outputs }} à pergunta {{ inputs }} é cordial, "
        "está em português e não é grosseira. Responda True se sim e False se não."
    ),
    model=JUIZ,
    feedback_value_type=bool,
)

JUIZES = [
    Correctness(model=JUIZ),
    RelevanceToQuery(model=JUIZ),
    Safety(model=JUIZ),
    Guidelines(name="concisao", guidelines="A resposta deve ter no máximo 3 frases.", model=JUIZ),
    tom_atendimento,
]

[juiz.name for juiz in JUIZES]

['correctness', 'relevance_to_query', 'safety', 'concisao', 'tom_atendimento']

## 7. Rodando a avaliação

`mlflow.genai.evaluate` executa `responder` em cada linha do conjunto, aplica os juízes e grava tudo num run do experimento. O argumento `model_id` liga os resultados à versão do modelo registrada na seção 3.

In [8]:
resultado = mlflow.genai.evaluate(
    data=DATASET,
    predict_fn=responder,
    scorers=JUIZES,
    model_id=info.model_id,
)

print(f"Run da avaliação: {resultado.run_id}")

2026/11/16 22:51:15 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-2bde2db730ca40a28c5be790f17956eb
2026/11/16 22:51:15 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/11/16 22:51:15 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/11/16 22:51:15 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: aged-bird-189
  Run ID: affe0f8cb51d45c8b3de3d16857558b8

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

Run da avaliação: affe0f8cb51d45c8b3de3d16857558b8


## 8. Métricas

Cada juiz vira uma métrica com a média dos vereditos (`/mean`). Para os juízes booleanos, a média é a fração de respostas aprovadas.

In [9]:
metricas = pd.DataFrame.from_dict(resultado.metrics, orient="index", columns=["valor"])
display(metricas.round(2))

,valor
safety/mean,1.00
tom_atendimento/mean,1.00
relevance_to_query/mean,0.75
concisao/mean,1.00
correctness/mean,0.50


## 9. Resultado por pergunta

A tabela mostra o veredito de cada juiz para cada pergunta. O campo `request` traz a entrada que o agente recebeu.

In [10]:
def pergunta_da_linha(valor) -> str:
    if isinstance(valor, dict) and "messages" in valor:
        return valor["messages"][0]["content"]
    return str(valor)


df = resultado.result_df
colunas_valor = [coluna for coluna in df.columns if coluna.endswith("/value")]

tabela = df[colunas_valor].copy()
tabela.insert(0, "pergunta", df["request"].map(pergunta_da_linha))
display(tabela)

,pergunta,safety/value,tom_atendimento/value,relevance_to_query/value,concisao/value,correctness/value,expected_response/value
0,Posso devolver um produto depois de 10 dias?,yes,True,yes,yes,yes,Informa que o direito de arrependimento é de 7...
1,Quanto tempo leva para chegar uma entrega em C...,yes,True,yes,yes,no,Informa um prazo em dias úteis para o frete pa...
2,Quero trocar um produto personalizado com grav...,yes,True,yes,yes,no,Informa que produtos personalizados não podem ...
3,Qual é a capital da França?,yes,True,no,yes,yes,Informa que só ajuda com devoluções e entregas...


## 10. Justificativas das reprovações

Para cada juiz, mostramos as respostas que ele reprovou e o motivo dado. A justificativa é o raciocínio do juiz, então vale ler antes de confiar no veredito.

In [11]:
for coluna in colunas_valor:
    nome_juiz = coluna.replace("/value", "")
    reprovadas = df[df[coluna].astype(str).str.lower().isin(["no", "false", "fail"])]
    if reprovadas.empty:
        print(f"{nome_juiz}: nenhuma reprovação.\n")
        continue
    print(f"{nome_juiz}: {len(reprovadas)} reprovação(ões)")
    for _, linha in reprovadas.iterrows():
        print(f"- {pergunta_da_linha(linha['request'])}")
        print(f"  Justificativa: {str(linha[f'{nome_juiz}/rationale'])[:400]}")
    print()

safety: nenhuma reprovação.

tom_atendimento: nenhuma reprovação.

relevance_to_query: 1 reprovação(ões)
- Qual é a capital da França?
  Justificativa: Let's think step by step: the question asks for the capital of France. The answer talks only about being able to help with returns and deliveries and that the question will be forwarded to an attendant. None of those statements provide any information about the capital of France, so the entire response is not relevant.

concisao: nenhuma reprovação.

correctness: 2 reprovação(ões)
- Quanto tempo leva para chegar uma entrega em Curitiba?
  Justificativa: Let's think step by step: The document does provide a delivery timeframe in business days for Curitiba (it states normally 2–7 dias úteis, with expresso 1–2 dias úteis), so the part about informing a timeframe for standard shipping is supported. The document asks the user to provide their CEP or to choose the shipping option in the cart to check the exact deadline for the region, so requ

## 11. Observações

- **Interface**: o experimento `agente-avaliacao` mostra o run da avaliação, com as métricas e os traces de cada pergunta. Com `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000` na pasta do notebook, abra `http://127.0.0.1:5000`.
- **Custo**: cada juiz faz uma chamada de LLM por linha. Com cinco juízes e quatro perguntas, são vinte chamadas de avaliação, além das invocações do agente.
- **Juiz e agente**: o juiz deve ser, de preferência, um modelo diferente ou mais forte que o agente. Um mesmo modelo pode favorecer as próprias respostas.
- **Qualidade do juiz**: os vereditos são estimativas. Vale conferir uma amostra à mão, sobretudo as reprovações, antes de tirar conclusões.
- **Contagem de traces**: o número mostrado na seção 5 inclui traces de execuções anteriores do mesmo experimento.
- **Reexecução**: cada execução cria um run de avaliação e uma nova versão do agente. Para recomeçar, apague `mlflow.db` e a pasta `prompts/`.